# Capstone: Compliance Document Reviewer

This notebook assembles every building block from the series into a single production-grade system: a **Compliance Document Reviewer** that accepts financial contract clauses, redacts personally identifiable information (PII), classifies each clause via a LangGraph multi-agent pipeline, and exposes the result through a FastAPI endpoint backed by observability, semantic caching, and intelligent model routing.

The system integrates: the RAG retrieval patterns from [notebook 07](/courses/llm-eng/07-rag-pipeline.html), the LangGraph agent architecture from [notebook 08](/courses/llm-eng/08-agents-langgraph.html), the evaluation harness from [notebook 04](/courses/llm-eng/04-eval-concepts.html), the cost optimisation stack from [notebook 11](/courses/llm-eng/11-cost-optimization.html), and the serving and infrastructure patterns from [notebooks 10](/courses/llm-eng/10-model-serving.html) and [12](/courses/llm-eng/12-ai-infra.html). The capstone is intentionally end-to-end: we write, test, and demonstrate every layer.

Setup:

In [ ]:
#| echo: false
import os, json, re, time, hashlib
import numpy as np
from dotenv import load_dotenv
load_dotenv()

import openai
from pydantic import BaseModel
from typing import Optional, Type

PRICES = {
    "gpt-4o":      {"input": 2.50,  "output": 10.00},
    "gpt-4o-mini": {"input": 0.15,  "output": 0.60},
}

class LLMClient:
    def __init__(self, model="gpt-4o-mini", temperature=0.0):
        self.model = model; self.temperature = temperature
        self._client = openai.OpenAI()
        self._in = 0; self._out = 0

    def complete(self, messages, *, response_format=None):
        if response_format is not None:
            resp = self._client.beta.chat.completions.parse(
                model=self.model, messages=messages,
                temperature=self.temperature, response_format=response_format)
        else:
            resp = self._client.chat.completions.create(
                model=self.model, messages=messages, temperature=self.temperature)
        if resp.usage:
            self._in += resp.usage.prompt_tokens; self._out += resp.usage.completion_tokens
        if response_format is not None: return resp.choices[0].message.parsed
        return resp.choices[0].message.content

    @property
    def total_cost(self):
        if self.model not in PRICES: return 0.0
        p = PRICES[self.model]
        return (self._in * p["input"] + self._out * p["output"]) / 1_000_000

llm = LLMClient()

## PII Redaction

**Why redact before sending to the LLM.** Financial contracts contain names, account numbers, tax identifiers, and email addresses that are PII under GDPR, CCPA, and various financial privacy regulations. Sending raw contract text to a third-party LLM API risks violating data processing agreements. The standard mitigation is **pre-processing redaction**: replace PII with typed placeholders before the text leaves the organisation's boundary, process with the LLM, then optionally re-hydrate the placeholders in the response.

<br>

**Microsoft Presidio.** [Presidio](https://microsoft.github.io/presidio/) is the de facto open-source library for PII detection and anonymisation. It ships with recognisers for names, email addresses, phone numbers, credit card numbers, US SSNs, IBANs, and many other entity types. It uses a combination of regular expressions, named-entity recognition (a spaCy NER model), and a rule-based denial list. We use the `AnalyzerEngine` to identify PII spans and the `AnonymizerEngine` to replace them with `<ENTITY_TYPE>` placeholders.

<br>

**Re-hydration.** After the LLM returns its analysis, we may want to replace `<PERSON>` back with the original name in the output report. Presidio's `DeanonymizeEngine` supports this via an `OperatorConfig` map that records the original values. We implement both the redaction and re-hydration functions below.

Installing and configuring Presidio:

In [ ]:
#| echo: false
# Presidio requires spaCy's en_core_web_lg model.
# Run once: python -m spacy download en_core_web_lg
try:
    from presidio_analyzer   import AnalyzerEngine
    from presidio_anonymizer import AnonymizerEngine
    from presidio_anonymizer.entities import OperatorConfig
    PRESIDIO_AVAILABLE = True
except ImportError:
    PRESIDIO_AVAILABLE = False
    print("presidio not installed — PII redaction will use regex fallback")

Implementing the `PIIRedactor` with a regex fallback when Presidio is unavailable:

In [ ]:
import re
from dataclasses import dataclass, field

# Regex patterns for the fallback redactor
_EMAIL_RE = re.compile(r"[\w.+-]+@[\w-]+\.[a-z]{2,}", re.I)
_PHONE_RE = re.compile(r"\b(\+?1[\s.-]?)?\(?\d{3}\)?[\s.-]?\d{3}[\s.-]?\d{4}\b")
_SSN_RE   = re.compile(r"\b\d{3}-\d{2}-\d{4}\b")
_IBAN_RE  = re.compile(r"\b[A-Z]{2}\d{2}[A-Z0-9]{4}\d{7}([A-Z0-9]?){0,16}\b")

@dataclass
class RedactionResult:
    redacted_text: str
    mapping:       dict = field(default_factory=dict)  # placeholder → original

class PIIRedactor:
    """PII redactor with Presidio (primary) and regex (fallback)."""

    def __init__(self):
        if PRESIDIO_AVAILABLE:
            self._analyzer   = AnalyzerEngine()
            self._anonymizer = AnonymizerEngine()
        self._counter: dict[str, int] = {}

    def redact(self, text: str) -> RedactionResult:
        if PRESIDIO_AVAILABLE:
            return self._redact_presidio(text)
        return self._redact_regex(text)

    def _redact_presidio(self, text: str) -> RedactionResult:
        results = self._analyzer.analyze(text=text, language="en")  # <1>
        mapping: dict[str, str] = {}

        def replace_with_placeholder(entity):
            etype = entity.entity_type
            n     = self._counter.get(etype, 0) + 1
            self._counter[etype] = n
            ph    = f"<{etype}_{n}>"
            mapping[ph] = text[entity.start:entity.end]
            return ph

        ops     = {r.entity_type: OperatorConfig("replace", {"new_value": replace_with_placeholder(r)})
                   for r in results}
        anon    = self._anonymizer.anonymize(text=text, analyzer_results=results, operators=ops)
        return RedactionResult(redacted_text=anon.text, mapping=mapping)

    def _redact_regex(self, text: str) -> RedactionResult:  # <2>
        mapping: dict[str, str] = {}

        def _replace(pattern, label, t):
            def sub(m):
                n  = sum(1 for k in mapping if k.startswith(f"<{label}_")) + 1
                ph = f"<{label}_{n}>"
                mapping[ph] = m.group(0)
                return ph
            return pattern.sub(sub, t)

        text = _replace(_EMAIL_RE, "EMAIL",  text)
        text = _replace(_PHONE_RE, "PHONE",  text)
        text = _replace(_SSN_RE,   "SSN",    text)
        text = _replace(_IBAN_RE,  "IBAN",   text)
        return RedactionResult(redacted_text=text, mapping=mapping)

    def rehydrate(self, text: str, mapping: dict) -> str:
        for placeholder, original in mapping.items():
            text = text.replace(placeholder, original)
        return text

1. `analyzer.analyze` returns a list of `RecognizerResult` objects, each with `entity_type`, `start`, `end`, and `score`. We iterate over them to build a placeholder mapping before anonymisation.
2. The regex fallback covers the four most common financial PII types — sufficient for testing without the Presidio dependency.

Testing the redactor on a sample contract clause:

In [ ]:
redactor = PIIRedactor()

sample_clause = (
    "This agreement is entered into by John Martinez (SSN: 123-45-6789, "
    "email: j.martinez@hedgefund.com, phone: +1-212-555-0147). "
    "Counterparty account IBAN: GB82WEST12345698765432. "
    "The party agrees to maintain a minimum margin of 25%."
)

result = redactor.redact(sample_clause)
print("Redacted:")
print(result.redacted_text)
print("\nMapping:", json.dumps(result.mapping, indent=2))

## LangGraph Compliance Pipeline

**Pipeline architecture.** The compliance pipeline has four nodes: (1) **Classify** — a fast GPT-4o-mini call that assigns one of five clause types (`CAPITAL_ADEQUACY`, `MARGIN`, `CREDIT_RISK`, `OPERATIONAL`, `OTHER`) and a risk score (1–5); (2) **Retrieve** — a ChromaDB lookup that fetches the three most relevant regulatory precedents for this clause type; (3) **Analyse** — a model-routed LLM call that produces a structured `ComplianceReport` with a verdict, confidence, and justification citing retrieved precedents; (4) **Review** — for high-risk (score ≥ 4) clauses, a GPT-4o call reviews the analysis output and can escalate or override the verdict.

<br>

**State type.** The pipeline state is a `TypedDict` carrying: the original clause, its redacted version, the classification result, the retrieved context, and the final report. All nodes read from and write to this shared dict. LangGraph's graph engine ensures nodes run in dependency order and re-runs any node whose upstream state has changed.

<br>

**Conditional routing.** After the Analyse node, a conditional edge checks `state["classification"].risk_score`. Scores ≥ 4 route to Review; scores < 4 route directly to END. This mirrors the pattern from [notebook 08](/courses/llm-eng/08-agents-langgraph.html) but integrates PII redaction and RAG retrieval as first-class pipeline stages.

Defining the Pydantic schemas and `TypedDict` state:

In [ ]:
from typing import TypedDict, Literal
from pydantic import BaseModel, Field

ClauseType = Literal["CAPITAL_ADEQUACY", "MARGIN", "CREDIT_RISK", "OPERATIONAL", "OTHER"]
Verdict    = Literal["COMPLIANT", "NON_COMPLIANT", "REQUIRES_REVIEW"]

class Classification(BaseModel):
    clause_type: ClauseType
    risk_score:  int = Field(ge=1, le=5, description="1=low, 5=critical")
    rationale:   str

class ComplianceReport(BaseModel):
    verdict:      Verdict
    confidence:   float = Field(ge=0.0, le=1.0)
    justification: str
    citations:    list[str] = Field(default_factory=list)
    escalated:    bool      = False

class PipelineState(TypedDict):
    clause:         str                # original (may contain PII)
    redacted:       str                # PII-redacted version
    pii_mapping:    dict               # placeholder → original
    classification: Classification | None
    context:        list[str]          # retrieved regulatory passages
    report:         ComplianceReport | None

Defining the four pipeline node functions:

In [ ]:
# Minimal regulatory knowledge base (in production this is ChromaDB)
REGULATORY_KB = {
    "CAPITAL_ADEQUACY": [
        "Basel III requires a minimum CET1 ratio of 4.5% and total capital ratio of 8.0%.",
        "SIFI surcharges add 1–3.5% to CET1 requirements for systemically important firms.",
        "Capital buffers (conservation: 2.5%, countercyclical: 0–2.5%) apply above minimums.",
    ],
    "MARGIN": [
        "EMIR requires daily variation margin and initial margin for OTC derivatives above threshold.",
        "Reg T sets an initial margin requirement of 50% for equity purchases in US markets.",
        "Margin calls must be met within one business day under standard prime brokerage agreements.",
    ],
    "CREDIT_RISK": [
        "Counterparty credit risk must be managed via netting agreements and collateral.",
        "CVA capital charge applies to mark-to-market losses on OTC derivative exposures.",
        "Credit limits must be approved by the credit committee and reviewed annually.",
    ],
    "OPERATIONAL": [
        "Operational risk capital under Basel III uses the Standardised Approach (SA-OPR).",
        "Business continuity plans must be tested at least annually.",
        "Incident reporting to regulators is required within 72 hours of a material breach.",
    ],
    "OTHER": [
        "General contract terms should conform to ISDA Master Agreement standards.",
        "Governing law and dispute resolution must be specified in financial contracts.",
    ],
}

redactor = PIIRedactor()
llm_fast  = LLMClient(model="gpt-4o-mini")
llm_strong = LLMClient(model="gpt-4o")

def node_classify(state: PipelineState) -> dict:
    """Node 1: redact PII, then classify clause type and risk."""
    result = redactor.redact(state["clause"])  # <1>
    classif = llm_fast.complete(
        messages=[
            {"role": "system", "content": (
                "Classify the financial contract clause. "
                "Return JSON with clause_type (CAPITAL_ADEQUACY|MARGIN|CREDIT_RISK|OPERATIONAL|OTHER), "
                "risk_score (1-5, where 5=critical), and rationale."
            )},
            {"role": "user", "content": result.redacted_text},
        ],
        response_format=Classification,
    )
    return {
        "redacted":       result.redacted_text,
        "pii_mapping":    result.mapping,
        "classification": classif,
    }

def node_retrieve(state: PipelineState) -> dict:
    """Node 2: retrieve relevant regulatory passages for the clause type."""
    ctype   = state["classification"].clause_type
    context = REGULATORY_KB.get(ctype, REGULATORY_KB["OTHER"])[:3]  # <2>
    return {"context": context}

def node_analyse(state: PipelineState) -> dict:
    """Node 3: produce compliance verdict using retrieved context."""
    ctx_str = "\n".join(f"- {c}" for c in state["context"])
    report = llm_fast.complete(
        messages=[
            {"role": "system", "content": (
                f"You are a compliance analyst. Regulatory context:\n{ctx_str}\n\n"
                "Classify the clause as COMPLIANT, NON_COMPLIANT, or REQUIRES_REVIEW. "
                "Return JSON: verdict, confidence (0-1), justification, citations (list of relevant rules)."
            )},
            {"role": "user", "content": state["redacted"]},
        ],
        response_format=ComplianceReport,
    )
    return {"report": report}

def node_review(state: PipelineState) -> dict:
    """Node 4 (high-risk only): senior review with GPT-4o, may escalate."""
    existing = state["report"]
    review = llm_strong.complete(  # <3>
        messages=[
            {"role": "system", "content": (
                "You are a senior compliance officer reviewing a high-risk clause. "
                f"A junior analyst produced: verdict={existing.verdict}, "
                f"confidence={existing.confidence:.2f}, justification={existing.justification}. "
                "You may confirm or override. If overriding, set escalated=true. "
                "Return JSON: verdict, confidence, justification, citations, escalated."
            )},
            {"role": "user", "content": state["redacted"]},
        ],
        response_format=ComplianceReport,
    )
    return {"report": review}

1. PII redaction happens at the very first node — the redacted text flows through all subsequent nodes, and the raw `clause` string never leaves the application boundary.
2. In production this is a ChromaDB ANN lookup; here we use a static dict keyed by clause type for reproducibility.
3. The review node uses `llm_strong` (GPT-4o) rather than `llm_fast` — this cost is incurred only for high-risk clauses (risk_score ≥ 4), keeping the average cost close to GPT-4o-mini.

Assembling the LangGraph state machine:

In [ ]:
try:
    from langgraph.graph import StateGraph, END
    LANGGRAPH_AVAILABLE = True
except ImportError:
    LANGGRAPH_AVAILABLE = False

def build_pipeline():
    if not LANGGRAPH_AVAILABLE:
        return None

    graph = StateGraph(PipelineState)

    graph.add_node("classify", node_classify)
    graph.add_node("retrieve", node_retrieve)
    graph.add_node("analyse",  node_analyse)
    graph.add_node("review",   node_review)

    graph.set_entry_point("classify")
    graph.add_edge("classify", "retrieve")
    graph.add_edge("retrieve", "analyse")

    def route_after_analyse(state: PipelineState) -> str:  # <1>
        score = state["classification"].risk_score
        return "review" if score >= 4 else END

    graph.add_conditional_edges("analyse", route_after_analyse,
                                {"review": "review", END: END})
    graph.add_edge("review", END)

    return graph.compile()

pipeline = build_pipeline()
if pipeline:
    print("Pipeline compiled successfully")
else:
    print("LangGraph not available — pipeline will run in fallback mode")

1. The routing function returns either `"review"` or `END` (a special LangGraph constant). The `add_conditional_edges` call maps each possible return value to a node name or `END` — a type-safe alternative to returning raw node names.

A fallback sequential runner for environments without LangGraph:

In [ ]:
def run_pipeline_fallback(clause: str) -> PipelineState:
    """Sequential fallback that mirrors the LangGraph pipeline."""
    state: PipelineState = {
        "clause": clause, "redacted": "", "pii_mapping": {},
        "classification": None, "context": [], "report": None,
    }
    state.update(node_classify(state))
    state.update(node_retrieve(state))
    state.update(node_analyse(state))
    if state["classification"].risk_score >= 4:
        state.update(node_review(state))
    return state

def run_pipeline(clause: str) -> PipelineState:
    if pipeline:
        return pipeline.invoke({
            "clause": clause, "redacted": "", "pii_mapping": {},
            "classification": None, "context": [], "report": None,
        })
    return run_pipeline_fallback(clause)

## End-to-End Demo

We now run the pipeline on five representative contract clauses drawn from different risk tiers, verifying that (1) PII is redacted before processing, (2) the correct clause type is identified, (3) high-risk clauses trigger the senior review node, and (4) the compliance verdict and citations are structurally valid.

Test clauses covering all five clause types at varying risk levels:

In [ ]:
TEST_CLAUSES = [
    # (label, clause)
    ("Low-risk margin",
     "The party agrees to maintain an initial margin of 60% on all equity positions, "
     "exceeding the standard Reg T requirement of 50%."),

    ("High-risk capital (with PII)",
     "As agreed with John Martinez (j.martinez@hedgefund.com), the counterparty commits "
     "to maintaining a CET1 ratio of only 3.8%, which management considers adequate."),

    ("Medium-risk credit",
     "Credit exposure to any single counterparty shall not exceed 15% of total risk-weighted "
     "assets. Exceptions require written approval from the Chief Risk Officer."),

    ("High-risk operational",
     "In the event of a cybersecurity breach, the firm is under no obligation to notify "
     "regulators unless losses exceed $10 million. Internal incident logs are optional."),

    ("Low-risk other",
     "This agreement is governed by the laws of England and Wales. Any dispute shall be "
     "referred to arbitration under the LCIA rules."),
]

Running all five clauses through the pipeline:

In [ ]:
results = []
for label, clause in TEST_CLAUSES:
    print(f"\n{'='*60}")
    print(f"Clause: {label}")
    state = run_pipeline(clause)
    c = state["classification"]
    r = state["report"]
    print(f"  Type:       {c.clause_type}  (risk={c.risk_score})")
    print(f"  PII redacted: {bool(state['pii_mapping'])} — placeholders: {list(state['pii_mapping'].keys())}")
    print(f"  Verdict:    {r.verdict}  (conf={r.confidence:.2f}  escalated={r.escalated})")
    print(f"  Citations:  {r.citations[:2]}")
    results.append({"label": label, **c.model_dump(), **r.model_dump()})

Summary table:

In [ ]:
#| code-fold: true
print(f"\n{'Label':<30} {'Type':<20} {'Risk':>4} {'Verdict':<18} {'Conf':>5} {'Esc'}")  
print("-" * 85)
for r in results:
    print(f"{r['label']:<30} {r['clause_type']:<20} {r['risk_score']:>4} "
          f"{r['verdict']:<18} {r['confidence']:>5.2f} {r['escalated']}")

## FastAPI Integration

**Wrapping the pipeline.** The compliance pipeline is a pure Python function — wrapping it in a FastAPI endpoint is straightforward. The POST `/review` endpoint accepts a `ReviewRequest` with a `clause` string and returns a `ReviewResponse` that mirrors the `ComplianceReport` schema plus the `clause_type` and `risk_score` from the classification step.

<br>

**Caching layer.** We integrate the `SemanticCache` from [notebook 11](/courses/llm-eng/11-cost-optimization.html) in front of the pipeline. The cache key is the *original* clause text (before PII redaction). This is safe because the cache is in-process and on-premises — the raw text never leaves the boundary via the cache path. Cache hits skip the entire pipeline and return the previously computed report in milliseconds.

<br>

**Async execution.** The pipeline makes synchronous OpenAI calls inside node functions. To avoid blocking FastAPI's event loop, we run the pipeline in a thread pool executor via `asyncio.to_thread`. This allows the endpoint to handle concurrent requests without each LLM call serialising the server.

The FastAPI app definition for the compliance service:

```python
# src/compliance_api.py
import asyncio
from contextlib import asynccontextmanager
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel

class ReviewRequest(BaseModel):
    clause: str

class ReviewResponse(BaseModel):
    clause_type:   str
    risk_score:    int
    verdict:       str
    confidence:    float
    justification: str
    citations:     list[str]
    escalated:     bool
    cache_hit:     bool = False

class AppState:
    pipeline = None
    cache    = None

app_state = AppState()

@asynccontextmanager
async def lifespan(app: FastAPI):
    app_state.pipeline = build_pipeline()     # warm up graph at startup  # <1>
    app_state.cache    = SemanticCache(threshold=0.92)
    yield

app = FastAPI(title="Compliance Reviewer", lifespan=lifespan)

@app.get("/health")
async def health():
    return {"status": "ok", "cache_entries": len(app_state.cache._store)}

@app.post("/review", response_model=ReviewResponse)
async def review(req: ReviewRequest):
    cached = app_state.cache.get(req.clause)  # <2>
    if cached:
        data = json.loads(cached)
        return ReviewResponse(**data, cache_hit=True)

    # Run pipeline in thread pool to avoid blocking the event loop
    state = await asyncio.to_thread(run_pipeline, req.clause)  # <3>

    c = state["classification"]
    r = state["report"]
    response = ReviewResponse(
        clause_type=c.clause_type, risk_score=c.risk_score,
        verdict=r.verdict, confidence=r.confidence,
        justification=r.justification, citations=r.citations,
        escalated=r.escalated,
    )
    # Cache the result as JSON for future similar queries
    app_state.cache.put(req.clause, response.model_dump_json())
    return response
```

1. `build_pipeline()` compiles the LangGraph state machine at startup — this parses the graph structure and validates all edges. Without warm-up, the first request would incur this overhead.
2. The semantic cache is checked with the original (un-redacted) clause text. If a near-duplicate clause was already reviewed, the result is returned immediately with `cache_hit=True` and zero LLM cost.
3. `asyncio.to_thread` runs `run_pipeline` in a `ThreadPoolExecutor` thread, freeing the event loop to handle other requests while waiting for the LLM response. This is the correct pattern for wrapping synchronous I/O-bound functions in FastAPI async handlers.

:::{.callout-note}
For a streaming compliance review (useful when the justification is long), replace `asyncio.to_thread` with an async generator that yields partial results using FastAPI's `StreamingResponse`. The `ComplianceReport.justification` field streams character-by-character while `verdict` and `confidence` are returned at the end.

:::

## Evaluation

**What to evaluate.** The compliance system has three layers to evaluate: (1) **PII redaction recall** — what fraction of PII entities were correctly redacted (false negatives expose sensitive data); (2) **classification accuracy** — are clauses assigned to the correct type and risk tier (wrong type → wrong regulatory context → wrong verdict); (3) **verdict accuracy** — on a labelled set, what fraction of verdicts match expert human judgement. We evaluate all three with a compact but representative test harness.

Defining the evaluation dataset and running it:

In [ ]:
from dataclasses import dataclass

@dataclass
class EvalCase:
    clause:          str
    expected_type:   str
    expected_verdict: str
    has_pii:         bool

EVAL_DATASET: list[EvalCase] = [
    EvalCase(
        "Our CET1 ratio is 14.8%, well above Basel III's 4.5% minimum.",
        "CAPITAL_ADEQUACY", "COMPLIANT", False),
    EvalCase(
        "The counterparty (contact: sarah@globalbank.com) agrees to a CET1 of 3.2%.",
        "CAPITAL_ADEQUACY", "NON_COMPLIANT", True),
    EvalCase(
        "Initial margin of 55% will be maintained on equity positions per Reg T requirements.",
        "MARGIN", "COMPLIANT", False),
    EvalCase(
        "Margin calls may be settled within 5 business days at the client's discretion.",
        "MARGIN", "NON_COMPLIANT", False),
    EvalCase(
        "The firm disclaims all duty to report cybersecurity incidents to regulators.",
        "OPERATIONAL", "NON_COMPLIANT", False),
    EvalCase(
        "Business continuity plans are reviewed and tested semi-annually.",
        "OPERATIONAL", "COMPLIANT", False),
    EvalCase(
        "All disputes shall be resolved by arbitration under LCIA rules, English law applies.",
        "OTHER", "COMPLIANT", False),
    EvalCase(
        "Single counterparty exposure is capped at 10% of RWA with CRO sign-off for exceptions.",
        "CREDIT_RISK", "COMPLIANT", False),
]

Running the eval harness and printing metrics:

In [ ]:
type_correct    = 0
verdict_correct = 0
pii_redacted    = 0
pii_cases       = 0

for case in EVAL_DATASET:
    state = run_pipeline(case.clause)
    pred_type    = state["classification"].clause_type
    pred_verdict = state["report"].verdict
    has_redaction = bool(state["pii_mapping"])

    type_correct    += (pred_type    == case.expected_type)
    verdict_correct += (pred_verdict == case.expected_verdict)
    if case.has_pii:
        pii_cases    += 1
        pii_redacted += has_redaction

n = len(EVAL_DATASET)
print(f"Classification accuracy: {type_correct}/{n}  = {type_correct/n:.0%}")
print(f"Verdict accuracy:        {verdict_correct}/{n} = {verdict_correct/n:.0%}")
if pii_cases:
    print(f"PII redaction recall:    {pii_redacted}/{pii_cases} = {pii_redacted/pii_cases:.0%}")

## Deployment Walkthrough

**Putting it all together.** The full production deployment stack uses every pattern from the series: the Dockerfile from [notebook 10](/courses/llm-eng/10-model-serving.html), the Terraform ECS/ECR infrastructure from [notebook 12](/courses/llm-eng/12-ai-infra.html), the GitHub Actions CI/CD pipeline from [notebook 12](/courses/llm-eng/12-ai-infra.html), the Langfuse observability from [notebook 06](/courses/llm-eng/06-observability.html), and the cost optimisation stack from [notebook 11](/courses/llm-eng/11-cost-optimization.html). The table below shows how each component maps to a notebook and the directory in the repository where it lives.

<br>

**Deployment sequence.** We follow this order: (1) `terraform init && terraform apply` to provision ECR, ECS cluster, and Secrets Manager; (2) `make secrets` to populate AWS Secrets Manager from the local `.env.prod` file; (3) push to `main` which triggers the GitHub Actions build-push-deploy pipeline; (4) verify the `/health` endpoint; (5) run the evaluation harness against the production endpoint; (6) enable Langfuse tracing and confirm traces appear in the dashboard. Steps 3–6 take roughly five minutes once the infrastructure is provisioned.

Component summary table:

| Component | Notebook | Path |
| :-- | :--: | :-- |
| FastAPI compliance service | [10](/courses/llm-eng/10-model-serving.html) | `src/compliance_api.py` |
| PII redaction (Presidio) | 13 (this) | `src/redactor.py` |
| LangGraph pipeline | [08](/courses/llm-eng/08-agents-langgraph.html) | `src/pipeline.py` |
| Semantic cache | [11](/courses/llm-eng/11-cost-optimization.html) | `src/cache.py` |
| Model router | [11](/courses/llm-eng/11-cost-optimization.html) | `src/router.py` |
| Langfuse observability | [06](/courses/llm-eng/06-observability.html) | `src/observability.py` |
| Docker multi-stage build | [10](/courses/llm-eng/10-model-serving.html) | `Dockerfile` |
| Terraform (ECR, ECS, SM) | [12](/courses/llm-eng/12-ai-infra.html) | `infra/` |
| GitHub Actions CI/CD | [12](/courses/llm-eng/12-ai-infra.html) | `.github/workflows/deploy.yml` |
| Evaluation harness | [04](/courses/llm-eng/04-eval-concepts.html) | `evals/compliance_eval.py` |

:::{.callout-important}
Before deploying to a financial services production environment, the PII redaction pipeline must be validated by a data privacy officer against the applicable regulation (GDPR, CCPA, or GLBA). Presidio's default models have known gaps for domain-specific identifiers (e.g. CUSIP, LEI, trading account numbers) that require custom recogniser plugins.

:::

## Exercises

1. **Add a custom Presidio recogniser.** Write a `PatternRecognizer` subclass that detects CUSIP identifiers (9-character alphanumeric codes, e.g. `037833100`). Register it with the `AnalyzerEngine` and verify that it correctly redacts a sample clause containing a CUSIP. Check that it does not produce false positives on 9-digit numbers that are not CUSIPs (e.g. ZIP codes).

2. **Implement re-hydration in the API.** Extend `ReviewResponse` with a `redacted_justification` field that contains the justification text with PII placeholders in place, and a `full_justification` field where placeholders have been replaced by original values using `redactor.rehydrate`. Expose both fields to the caller so downstream systems can choose the appropriate level of disclosure.

3. **Build an A/B test harness.** Implement a `ComplianceABTest` class that routes each incoming clause to either the `gpt-4o-mini` pipeline (control) or the `gpt-4o` pipeline (treatment) using a `random.random() < 0.2` flag. Log the verdict, confidence, cost, and latency for each group. After 50 clauses, compute whether the treatment group shows statistically significant improvement in verdict accuracy (use Fisher's exact test on the confusion matrices).

---

$\blacksquare$